# Diabetes Classification & Clustering
## Multi-Class Prediction of Diabetes Status

**Classes:** N = Non-diabetic | Y = Diabetic | P = Predict-diabetic

**Author:** Aananda Giri  
**Date:** 2026-05-19

## 0. Imports and Configuration

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             ConfusionMatrixDisplay, silhouette_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_score, completeness_score, v_measure_score,
                             davies_bouldin_score, calinski_harabasz_score)

# XGBoost
from xgboost import XGBClassifier

# Imbalanced learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Clustering & Dimensionality Reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting defaults
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

print('All imports successful.')

All imports successful.


## 1. Data Loading and Initial Inspection

In [2]:
# Load CSV — uses \\r line terminators (legacy Mac format)
df = pd.read_csv('Diabetes Dataset/Dataset of Diabetes .csv', lineterminator='\r')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

Shape: (1001, 14)
Columns: ['ID', 'No_Pation', 'Gender', 'AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL', 'LDL', 'VLDL', 'BMI', 'CLASS']


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,502,17975.0,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221.0,M,26.0,4.5,62.0,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975.0,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656.0,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223.0,M,33.0,7.1,46.0,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N
5,634,34224.0,F,45.0,2.3,24.0,4.0,2.9,1.0,1.0,1.5,0.4,21.0,N
6,721,34225.0,F,50.0,2.0,50.0,4.0,3.6,1.3,0.9,2.1,0.6,24.0,N
7,421,34227.0,M,48.0,4.7,47.0,4.0,2.9,0.8,0.9,1.6,0.4,24.0,N
8,670,34229.0,M,43.0,2.6,67.0,4.0,3.8,0.9,2.4,3.7,1.0,21.0,N
9,759,34230.0,F,32.0,3.6,28.0,4.0,3.8,2.0,2.4,3.8,1.0,24.0,N


In [3]:
# Data types and null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1001 non-null   str    
 1   No_Pation  1000 non-null   float64
 2   Gender     1000 non-null   str    
 3   AGE        1000 non-null   float64
 4   Urea       1000 non-null   float64
 5   Cr         1000 non-null   float64
 6   HbA1c      1000 non-null   float64
 7   Chol       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASS      1000 non-null   str    
dtypes: float64(11), str(3)
memory usage: 109.6 KB


In [4]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('No missing values found.')
else:
    print('Missing values:')
    print(missing)

Missing values:
No_Pation    1
Gender       1
AGE          1
Urea         1
Cr           1
HbA1c        1
Chol         1
TG           1
HDL          1
LDL          1
VLDL         1
BMI          1
CLASS        1
dtype: int64


In [5]:
# Drop identifier columns (ID and No_Pation are just row identifiers)
df = df.drop(columns=['ID', 'No_Pation'])
print(f'Shape after dropping IDs: {df.shape}')
df.head()

Shape after dropping IDs: (1001, 12)


,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,M,26.0,4.5,62.0,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,F,50.0,4.7,46.0,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,M,33.0,7.1,46.0,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [6]:
# Check for duplicates
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')
if dup_count > 0:
    df = df.drop_duplicates()
    print(f'After dropping duplicates, shape: {df.shape}')

# Strip whitespace from categorical columns
df['Gender'] = df['Gender'].str.strip()
df['CLASS'] = df['CLASS'].str.strip()

# Verify CLASS values
print(f'\nCLASS distribution:')
print(df['CLASS'].value_counts())
print(f'\nUnique Gender values: {df["Gender"].unique()}')
print(f'Unique CLASS values: {df["CLASS"].unique()}')

Duplicate rows: 169
After dropping duplicates, shape: (832, 12)

CLASS distribution:
CLASS
Y    695
N     96
P     40
Name: count, dtype: int64

Unique Gender values: <StringArray>
['F', 'M', 'f', nan]
Length: 4, dtype: str
Unique CLASS values: <StringArray>
['N', 'P', 'Y', nan]
Length: 4, dtype: str
